# Boosting Models

## Shane Waldron

In [20]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

In [3]:
# Load the dataset
df = pd.read_csv('data/train.csv')

In [4]:
# Encoding the target variable
df["Irrigation_Need"] = df["Irrigation_Need"].map({"High": 2,"Medium": 1, "Low": 0})

# Removing the id column as it is not useful for modeling
df.drop("id", axis=1, inplace=True)

In [5]:
# Separating numerical and categorical columns
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
# Removing the target variable from the list of numerical columns
numerical_cols.remove("Irrigation_Need")

In [6]:
# Identifying X and y and splitting the data into training and testing sets
X = df.drop("Irrigation_Need", axis=1)
y = df["Irrigation_Need"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [16]:
# One-hot encoding for categorical variables
preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='passthrough'
)

In [10]:
# Training the XGBoost model with Optuna for hyperparameter tuning


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),                                
        "tree_method": "hist",
        "eval_metric": "mlogloss",
        "n_jobs": -1,
        'random_state': 42,
    }
    
    xgb_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(**params))
    ])
    
    scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    study_name="xgb_local",
    storage="sqlite:///optuna_xgb.db",
    load_if_exists=True,
)

# study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best Hyperparameters:", best_params)


[I 2026-04-23 09:27:49,093] Using an existing study with name 'xgb_local' instead of creating a new one.


Best Hyperparameters: {'n_estimators': 1186, 'max_depth': 5, 'learning_rate': 0.07392909715570185, 'subsample': 0.8435448211257797, 'colsample_bytree': 0.556708915636543, 'min_child_weight': 6, 'reg_alpha': 4.5224790845332776e-05, 'reg_lambda': 1.9087633509163162}


I had to rerun the notebook but didn't want to spend another 40 minutes allowing the study to run with 20 more trials, the first 20 trials are part of a db within GitHub

In [11]:
# Fitting the XGBoost model with the best hyperparameters
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(**best_params))
])
xgb_model.fit(X_train, y_train)
# Evaluating the XGBoost model
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    110927
           1       0.98      0.98      0.98     71683
           2       0.97      0.92      0.94      6390

    accuracy                           0.98    189000
   macro avg       0.98      0.96      0.97    189000
weighted avg       0.98      0.98      0.98    189000



### Evaluating XGBoost
The XGBoost model is performing very strongly overall, with 98% accuracy and weighted F1 of 0.98, which shows excellent aggregate classification performance. It predicts classes 0 and 1 especially well, with precision and recall both around 0.98 to 0.99, while high irrigation need is clearly the hardest case: its precision remains high at 0.97, but recall drops to 0.92, meaning the model is missing a noticeable share of true high irrigation need observations. The gap between weighted and macro averages also suggests the overall result is helped by the larger class sizes, so the main area for improvement would be better sensitivity on the minority class.

In [ ]:
# Modeling with LightGBM and Optuna for hyperparameter tuning
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),                                
        "random_state": 42,
        "verbosity": -1,
    }
    
    lgbm_model = Pipeline(steps=[
        ('preprocessor', preprocessor.set_output(transform='pandas')),
        ('classifier', LGBMClassifier(**params))
    ])
    
    scores = cross_val_score(lgbm_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=1)
    return scores.mean()

study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lgbm_local",
    storage="sqlite:///optuna_lgbm.db",
    load_if_exists=True,
)
study_lgbm.optimize(objective_lgbm, n_trials=20)



[I 2026-04-23 09:34:22,407] Using an existing study with name 'lgbm_local' instead of creating a new one.
[I 2026-04-23 09:35:20,481] Trial 6 finished with value: 0.9844557823129252 and parameters: {'n_estimators': 583, 'max_depth': 4, 'learning_rate': 0.17601331790893576, 'subsample': 0.6920600572784571, 'colsample_bytree': 0.553382718062905, 'min_child_weight': 6, 'reg_alpha': 1.3742965233981878e-06, 'reg_lambda': 4.8043048587639196e-08}. Best is trial 3 with value: 0.9845668934240364.
[I 2026-04-23 09:36:20,475] Trial 7 finished with value: 0.9844353741496599 and parameters: {'n_estimators': 799, 'max_depth': 3, 'learning_rate': 0.1163376968250577, 'subsample': 0.6815455478670049, 'colsample_bytree': 0.6476526537453853, 'min_child_weight': 8, 'reg_alpha': 0.006544275824809701, 'reg_lambda': 5.497852940120272e-08}. Best is trial 3 with value: 0.9845668934240364.
[I 2026-04-23 09:38:51,529] Trial 8 finished with value: 0.9845419501133786 and parameters: {'n_estimators': 695, 'max_dept

Best Hyperparameters for LightGBM: {'n_estimators': 446, 'max_depth': 9, 'learning_rate': 0.07915652512273584, 'subsample': 0.8708288055826356, 'colsample_bytree': 0.6335002107093821, 'min_child_weight': 5, 'reg_alpha': 0.6743080206628911, 'reg_lambda': 5.0943285309110366e-05}


In [24]:
best_params_lgbm = study_lgbm.best_params
print("Best Hyperparameters for LightGBM:", best_params_lgbm)

Best Hyperparameters for LightGBM: {'n_estimators': 598, 'max_depth': 5, 'learning_rate': 0.04649136440168943, 'subsample': 0.5045146723760011, 'colsample_bytree': 0.5825385426562086}


In [ ]:
# Modeling with LightGBM and Optuna for hyperparameter tuning with a different set of hyperparameters (and trying stratified cv)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    
    lgbm_model = Pipeline(steps=[
        ('preprocessor', preprocessor.set_output(transform='pandas')),
        ('classifier', LGBMClassifier(**params))
    ])
    
    scores = cross_val_score(lgbm_model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=1)
    return scores.mean()

study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lgbm_local_2",
    storage="sqlite:///optuna_lgbm2.db",
    load_if_exists=True,
)
study_lgbm.optimize(objective_lgbm, n_trials=20)



[I 2026-04-23 10:33:58,778] A new study created in RDB with name: lgbm_local_2
[I 2026-04-23 10:36:25,162] Trial 0 finished with value: 0.8221746031746033 and parameters: {'n_estimators': 1187, 'max_depth': 7, 'learning_rate': 0.2514524514665574, 'subsample': 0.8799640941877094, 'colsample_bytree': 0.731224307939001}. Best is trial 0 with value: 0.8221746031746033.
[I 2026-04-23 10:38:41,782] Trial 1 finished with value: 0.9841383219954649 and parameters: {'n_estimators': 831, 'max_depth': 8, 'learning_rate': 0.08697660409656671, 'subsample': 0.5840198837206304, 'colsample_bytree': 0.8854588855541821}. Best is trial 1 with value: 0.9841383219954649.
[I 2026-04-23 10:41:38,946] Trial 2 finished with value: 0.9842244897959184 and parameters: {'n_estimators': 1062, 'max_depth': 6, 'learning_rate': 0.018852162954532955, 'subsample': 0.9040425797908023, 'colsample_bytree': 0.8956313287631177}. Best is trial 2 with value: 0.9842244897959184.
[I 2026-04-23 10:44:34,462] Trial 3 finished with 

Best Hyperparameters for LightGBM: {'n_estimators': 598, 'max_depth': 5, 'learning_rate': 0.04649136440168943, 'subsample': 0.5045146723760011, 'colsample_bytree': 0.5825385426562086}


In [23]:
best_params_lgbm2 = study_lgbm.best_params
print("Best Hyperparameters for LightGBM:", best_params_lgbm2)

Best Hyperparameters for LightGBM: {'n_estimators': 598, 'max_depth': 5, 'learning_rate': 0.04649136440168943, 'subsample': 0.5045146723760011, 'colsample_bytree': 0.5825385426562086}


In [25]:
# Fitting the first LightGBM model with the best hyperparameters
lgbm_model = Pipeline(steps=[
    ('preprocessor', preprocessor.set_output(transform='pandas')),
    ('classifier', LGBMClassifier(**best_params_lgbm))
])
lgbm_model.fit(X_train, y_train)
# Evaluating the LightGBM model
y_pred_lgbm = lgbm_model.predict(X_test)
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

LightGBM Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    110927
           1       0.98      0.98      0.98     71683
           2       0.97      0.92      0.94      6390

    accuracy                           0.98    189000
   macro avg       0.98      0.96      0.97    189000
weighted avg       0.98      0.98      0.98    189000



In [26]:
# Fitting the second LightGBM model with the best hyperparameters
lgbm_model = Pipeline(steps=[
    ('preprocessor', preprocessor.set_output(transform='pandas')),
    ('classifier', LGBMClassifier(**best_params_lgbm2))
])
lgbm_model.fit(X_train, y_train)
# Evaluating the LightGBM model
y_pred_lgbm = lgbm_model.predict(X_test)
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

LightGBM Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    110927
           1       0.98      0.98      0.98     71683
           2       0.97      0.92      0.94      6390

    accuracy                           0.98    189000
   macro avg       0.98      0.96      0.97    189000
weighted avg       0.98      0.98      0.98    189000



### LightGBM analysis
LightGBM performs essentially identically to XGBoost on this classification task, with the same overall accuracy, weighted F1, and class-level precision and recall across low, medium, and high irrigation need. Both models classify low and medium irrigation need extremely well, while high irrigation need is the most difficult category: precision remains strong, but recall is lower, meaning both models miss some truly high-need cases. So in practical terms, neither model has a performance advantage here, and the decision between them would likely come down to factors like training speed, tuning convenience, or how they behave on the test set.